# Feature Extraction Pipeline

This notebook is organized into three reusable blocks:
- Utils block
- Primary features block
- Secondary features block

## Feature Dictionary

### Identifiers (Join Keys, Not Model Features)

1. `review_id`: Unique review key.
2. `user_id`: Reviewer identifier.
3. `business_id`: Business identifier.

### Primary Features

1. `stars` (Review): Initial rating given by the user.
2. `useful` (Review): Useful-vote count on the review.
3. `funny` (Review): Funny-vote count on the review.
4. `cool` (Review): Cool-vote count on the review.
5. `is_open` (Business): Business active status (`1` open, `0` closed).
6. `review_count` (User): User total written reviews.
7. `fans` (User): User fan count.
8. `friends_count` (User): Number of friend IDs in `friends`.
9. `compliment_writer` (User): Writing-related compliment count.

### Secondary Features (With Calculations)

1. `reviewer_bias`: This identifies if a low rating is "standard" for that user or a genuine outlier for the business.
   Formula: $reviewer\_bias = user\_stars - average\_stars$

2. `early_rating_variance`: Rating dispersion for a business in its early window.
   The standard deviation of stars within the "early" window ($t_0$ to $t_{early}$). High variance suggests inconsistent service.

3. `early_review_velocity`: Launch momentum of a business.
   Formula: $early\_review\_velocity = N_{early\_reviews} / \Delta t_{days}$, where $\Delta t$ is days between first and last early review.

4. `user_tenure_days`: Time on platform when the review was written.
   Formula: $user\_tenure\_days = (review\_date - yelping\_since)$ in days.

5. `elite_persistence`: Long-term elite history.
   Calculation: number of entries in the `elite` list/string.

6. `review_char_count`: Metadata complexity by character length.
   Formula: $review\_char\_count = len(text)$

7. `review_word_count`: Metadata complexity by word length.
   Formula: $review\_word\_count = \text{number of whitespace-split tokens in } text$

8. `attribute_consistency`: Whether user history aligns with current business categories.
   Calculation: `True` if at least one category of the current business has appeared in the same user’s prior reviews (ordered by date), else `False`.

## Utils Block

In [ ]:
from __future__ import annotations

from pathlib import Path
import ast
from typing import Iterable

import pandas as pd


# Default file names in the Yelp dataset folder.
REVIEW_FILE = "yelp_academic_dataset_review.json"
BUSINESS_FILE = "yelp_academic_dataset_business.json"
USER_FILE = "yelp_academic_dataset_user.json"


def read_jsonl_columns(path: str | Path, columns: Iterable[str], nrows: int | None = None) -> pd.DataFrame:
    """Read selected columns from a JSONL file to keep memory usage controlled."""
    path = Path(path)
    df = pd.read_json(path, lines=True, nrows=nrows)
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns in {path.name}: {missing}")
    return df.loc[:, list(columns)].copy()


def _parse_bool_like(value: object) -> pd.BooleanDtype:
    """Convert common bool-like Yelp values into nullable booleans."""
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().lower()
    if text in {"true", "1", "yes", "y"}:
        return True
    if text in {"false", "0", "no", "n"}:
        return False
    return pd.NA


def _friends_to_count(value: object) -> int:
    """Convert Yelp friends field into a numeric friend count."""
    if pd.isna(value) or value in (None, "", "None"):
        return 0
    if isinstance(value, list):
        return len(value)
    if isinstance(value, str):
        items = [x.strip() for x in value.split(",") if x.strip()]
        return len(items)
    return 0


def _split_csv_like(value: object) -> list[str]:
    """Split comma-separated Yelp fields into a clean list."""
    if pd.isna(value) or value in (None, "", "None"):
        return []
    if isinstance(value, list):
        return [str(v).strip() for v in value if str(v).strip()]
    return [v.strip() for v in str(value).split(",") if v.strip()]


def _list_length(value: object) -> int:
    """Count items for list-like Yelp string/list fields."""
    return len(_split_csv_like(value))


def filter_businesses_by_min_history_days(
    review_df: pd.DataFrame,
    min_days: int = 365,
    business_col: str = "business_id",
    date_col: str = "date",
) -> pd.DataFrame:
    """Keep only businesses whose review timeline spans at least `min_days` days."""
    required = [business_col, date_col]
    missing = [c for c in required if c not in review_df.columns]
    if missing:
        raise KeyError(f"review_df missing columns: {missing}")

    temp = review_df.loc[:, [business_col, date_col]].copy()
    temp[date_col] = pd.to_datetime(temp[date_col], errors="coerce")
    temp = temp.dropna(subset=[date_col])

    spans = temp.groupby(business_col, as_index=False).agg(
        first_date=(date_col, "min"),
        last_date=(date_col, "max"),
    )
    spans["history_days"] = (spans["last_date"] - spans["first_date"]).dt.days

    keep_ids = spans.loc[spans["history_days"] >= min_days, business_col]
    return review_df[review_df[business_col].isin(keep_ids)].copy()

## Primary Features Block

Main entry point:
- `extract_primary_features(...)`

In [ ]:
def build_review_primary_features(review_df: pd.DataFrame) -> pd.DataFrame:
    """Primary features from review records."""
    required = ["review_id", "user_id", "business_id", "stars", "useful", "funny", "cool"]
    missing = [c for c in required if c not in review_df.columns]
    if missing:
        raise KeyError(f"review_df missing columns: {missing}")
    return review_df.loc[:, required].copy()


def build_business_primary_features(business_df: pd.DataFrame) -> pd.DataFrame:
    """Primary features from business records"""
    required = ["business_id", "is_open"]
    missing = [c for c in required if c not in business_df.columns]
    if missing:
        raise KeyError(f"business_df missing columns: {missing}")

    out = business_df.loc[:, required].copy()
    out["is_open"] = out["is_open"].astype("Int64")

    return out


def build_user_primary_features(user_df: pd.DataFrame) -> pd.DataFrame:
    """Primary features from user records."""
    required = ["user_id", "review_count", "fans", "friends", "compliment_writer"]
    missing = [c for c in required if c not in user_df.columns]
    if missing:
        raise KeyError(f"user_df missing columns: {missing}")

    out = user_df.loc[:, required].copy()
    out["friends"] = out["friends"].apply(_friends_to_count).astype("Int64")
    out = out.rename(columns={"friends": "friends_count"})
    return out


def extract_primary_features(
    data_dir: str | Path,
    review_df: pd.DataFrame | None = None,
    business_df: pd.DataFrame | None = None,
    user_df: pd.DataFrame | None = None,
    nrows: int | None = None,
) -> pd.DataFrame:
    """Main function: extract and merge primary features directly from Yelp raw datasets."""
    data_dir = Path(data_dir)

    if review_df is None:
        review_df = read_jsonl_columns(
            data_dir / REVIEW_FILE,
            ["review_id", "user_id", "business_id", "stars", "useful", "funny", "cool"],
            nrows=nrows,
        )
    if business_df is None:
        business_df = read_jsonl_columns(
            data_dir / BUSINESS_FILE,
            ["business_id", "is_open"],
            nrows=nrows,
        )
    if user_df is None:
        user_df = read_jsonl_columns(
            data_dir / USER_FILE,
            ["user_id", "review_count", "fans", "friends", "compliment_writer"],
            nrows=nrows,
        )

    review_features = build_review_primary_features(review_df)
    business_features = build_business_primary_features(business_df)
    user_features = build_user_primary_features(user_df)

    merged = review_features.merge(
        business_features,
        on="business_id",
        how="left",
        validate="m:1",
    )
    merged = merged.merge(
        user_features,
        on="user_id",
        how="left",
        validate="m:1",
    )
    return merged

## Secondary Features Block

Main entry point:
- `extract_secondary_features(...)`

In [4]:
def _build_early_window_metrics(review_df: pd.DataFrame, early_window_days: int) -> pd.DataFrame:
    """Compute early-window aggregates used by multiple features."""
    temp = review_df.loc[:, ["business_id", "date", "stars"]].copy()
    temp["date"] = pd.to_datetime(temp["date"], errors="coerce")
    temp = temp.dropna(subset=["date"])

    business_start = temp.groupby("business_id", as_index=False)["date"].min().rename(
        columns={"date": "business_t0"}
    )
    temp = temp.merge(business_start, on="business_id", how="left", validate="m:1")

    temp["days_since_t0"] = (temp["date"] - temp["business_t0"]).dt.days
    early = temp[temp["days_since_t0"].between(0, early_window_days)]

    agg = early.groupby("business_id", as_index=False).agg(
        early_rating_variance=("stars", "std"),
        early_review_count=("stars", "count"),
        early_first_date=("date", "min"),
        early_last_date=("date", "max"),
    )

    span_days = (agg["early_last_date"] - agg["early_first_date"]).dt.days.fillna(0)
    agg["early_review_velocity"] = agg["early_review_count"] / span_days.clip(lower=1)
    return agg.loc[:, ["business_id", "early_rating_variance", "early_review_velocity"]]


def _build_attribute_consistency_feature(review_df: pd.DataFrame, business_df: pd.DataFrame,) -> pd.DataFrame:
    """Build attribute_consistency feature per review."""
    categories_by_business = business_df.loc[:, ["business_id", "categories"]].copy()
    categories_by_business["categories"] = categories_by_business["categories"].apply(_split_csv_like)

    temp = review_df.loc[:, ["review_id", "user_id", "business_id", "date"]].copy()
    temp["date"] = pd.to_datetime(temp["date"], errors="coerce")
    temp = temp.merge(categories_by_business, on="business_id", how="left", validate="m:1")
    temp["categories"] = temp["categories"].apply(lambda v: v if isinstance(v, list) else [])

    exploded = temp.explode("categories")
    exploded = exploded.dropna(subset=["categories", "date"])
    exploded = exploded.sort_values(["user_id", "date", "review_id"])

    exploded["seen_before_same_category"] = (
        exploded.groupby(["user_id", "categories"]).cumcount() > 0
    )

    out = exploded.groupby("review_id", as_index=False)["seen_before_same_category"].max()
    out = out.rename(columns={"seen_before_same_category": "attribute_consistency"})
    return out


def _build_reviewer_bias_feature(out: pd.DataFrame) -> pd.Series:
    """Feature: reviewer_bias = stars - average_stars."""
    return out["stars"] - out["average_stars"]


def _build_user_tenure_days_feature(out: pd.DataFrame) -> pd.Series:
    """Feature: user_tenure_days from review date and yelping_since."""
    return (out["date"] - out["yelping_since"]).dt.days.astype("Int64")


def _build_elite_persistence_feature(user_df: pd.DataFrame) -> pd.DataFrame:
    """Feature: elite_persistence from the elite field."""
    user_min = user_df.loc[:, ["user_id", "average_stars", "yelping_since", "elite"]].copy()
    user_min["yelping_since"] = pd.to_datetime(user_min["yelping_since"], errors="coerce")
    user_min["elite_persistence"] = user_min["elite"].apply(_list_length).astype("Int64")
    return user_min


def _build_review_char_count_feature(out: pd.DataFrame) -> pd.Series:
    """Feature: character count of review text."""
    text_series = out["text"].fillna("").astype(str)
    return text_series.str.len().astype("Int64")


def _build_review_word_count_feature(out: pd.DataFrame) -> pd.Series:
    """Feature: word count of review text."""
    text_series = out["text"].fillna("").astype(str)
    return text_series.str.split().str.len().astype("Int64")


def _build_early_rating_variance_feature(review_df: pd.DataFrame, early_window_days: int) -> pd.DataFrame:
    """Feature: early_rating_variance by business_id."""
    metrics = _build_early_window_metrics(review_df=review_df, early_window_days=early_window_days)
    return metrics.loc[:, ["business_id", "early_rating_variance"]]


def _build_early_review_velocity_feature(review_df: pd.DataFrame, early_window_days: int) -> pd.DataFrame:
    """Feature: early_review_velocity by business_id."""
    metrics = _build_early_window_metrics(review_df=review_df, early_window_days=early_window_days)
    return metrics.loc[:, ["business_id", "early_review_velocity"]]


def build_secondary_features(
    review_df: pd.DataFrame,
    business_df: pd.DataFrame,
    user_df: pd.DataFrame,
    early_window_days: int = 30,
) -> pd.DataFrame:
    """Create level-2 secondary features from merged/derived information."""
    review_required = ["review_id", "user_id", "business_id", "date", "stars", "text"]
    business_required = ["business_id", "categories"]
    user_required = ["user_id", "average_stars", "yelping_since", "elite"]

    miss_review = [c for c in review_required if c not in review_df.columns]
    miss_business = [c for c in business_required if c not in business_df.columns]
    miss_user = [c for c in user_required if c not in user_df.columns]
    if miss_review:
        raise KeyError(f"review_df missing columns: {miss_review}")
    if miss_business:
        raise KeyError(f"business_df missing columns: {miss_business}")
    if miss_user:
        raise KeyError(f"user_df missing columns: {miss_user}")

    out = review_df.loc[:, ["review_id", "user_id", "business_id", "date", "stars", "text"]].copy()
    out["date"] = pd.to_datetime(out["date"], errors="coerce")

    user_min = _build_elite_persistence_feature(user_df)
    out = out.merge(user_min, on="user_id", how="left", validate="m:1")

    # One function call per feature.
    out["reviewer_bias"] = _build_reviewer_bias_feature(out)
    out["user_tenure_days"] = _build_user_tenure_days_feature(out)
    out["review_char_count"] = _build_review_char_count_feature(out)
    out["review_word_count"] = _build_review_word_count_feature(out)

    early_var = _build_early_rating_variance_feature(review_df=review_df, early_window_days=early_window_days)
    out = out.merge(early_var, on="business_id", how="left", validate="m:1")

    early_vel = _build_early_review_velocity_feature(review_df=review_df, early_window_days=early_window_days)
    out = out.merge(early_vel, on="business_id", how="left", validate="m:1")

    attr_consistency = _build_attribute_consistency_feature(review_df=review_df, business_df=business_df)
    out = out.merge(attr_consistency, on="review_id", how="left", validate="1:1")
    out["attribute_consistency"] = out["attribute_consistency"].fillna(False).astype(bool)

    secondary_cols = [
        "review_id",
        "reviewer_bias",
        "early_rating_variance",
        "early_review_velocity",
        "user_tenure_days",
        "elite_persistence",
        "review_char_count",
        "review_word_count",
        "attribute_consistency",
    ]
    return out.loc[:, secondary_cols]


def extract_secondary_features(
    data_dir: str | Path,
    review_df: pd.DataFrame | None = None,
    business_df: pd.DataFrame | None = None,
    user_df: pd.DataFrame | None = None,
    nrows: int | None = None,
    early_window_days: int = 30,
) -> pd.DataFrame:
    """Public secondary-feature extractor."""
    data_dir = Path(data_dir)

    if review_df is None:
        review_df = read_jsonl_columns(
            data_dir / REVIEW_FILE,
            [
                "review_id",
                "user_id",
                "business_id",
                "date",
                "stars",
                "text",
                "useful",
                "funny",
                "cool",
            ],
            nrows=nrows,
        )
    if business_df is None:
        business_df = read_jsonl_columns(
            data_dir / BUSINESS_FILE,
            ["business_id", "is_open", "attributes", "categories"],
            nrows=nrows,
        )
    if user_df is None:
        user_df = read_jsonl_columns(
            data_dir / USER_FILE,
            [
                "user_id",
                "review_count",
                "fans",
                "friends",
                "compliment_writer",
                "average_stars",
                "yelping_since",
                "elite",
            ],
            nrows=nrows,
        )

    return build_secondary_features(
        review_df=review_df,
        business_df=business_df,
        user_df=user_df,
        early_window_days=early_window_days,
    )

## Features Block

Main entry point:
- `extract_features(...)`

In [5]:
def extract_features(
    data_dir: str | Path,
    review_df: pd.DataFrame | None = None,
    business_df: pd.DataFrame | None = None,
    user_df: pd.DataFrame | None = None,
    nrows: int | None = None,
    early_window_days: int = 30,
) -> pd.DataFrame:
    """Public feature extractor that joins the primary and secondary feature sets."""
    primary = extract_primary_features(
        data_dir=data_dir,
        review_df=review_df,
        business_df=business_df,
        user_df=user_df,
        nrows=nrows,
    )
    secondary = extract_secondary_features(
        data_dir=data_dir,
        review_df=review_df,
        business_df=business_df,
        user_df=user_df,
        nrows=nrows,
        early_window_days=early_window_days,
    )
    return primary.merge(secondary, on="review_id", how="left", validate="1:1")

In [8]:
# Test Cell 1: Primary feature extraction on filtered data only (memory-safe)
DATA_DIR = Path("./yelp_dataset")
TEST_NROWS = None #20_000

review_sample_for_filter = read_jsonl_columns(
    DATA_DIR / REVIEW_FILE,
    ["review_id", "user_id", "business_id", "date", "stars", "useful", "funny", "cool", "text"],
    nrows=TEST_NROWS,
)
review_filtered = filter_businesses_by_min_history_days(
    review_df=review_sample_for_filter,
    min_days=365,
)

primary_features_sample = extract_primary_features(
    data_dir=DATA_DIR,
    review_df=review_filtered,
    nrows=TEST_NROWS,
)

print("Reviews before filter:", len(review_sample_for_filter))
print("Reviews after filter:", len(review_filtered))
print("Primary shape:", primary_features_sample.shape)
primary_preview_cols = [
    "review_id",
    "stars",
    "useful",
    "funny",
    "cool",
    "is_open",
    "review_count",
    "fans",
    "friends_count",
    "compliment_writer",
]
print(primary_features_sample[primary_preview_cols].head(5).to_string(index=False))

Reviews before filter: 6990280
Reviews after filter: 6897553
Primary shape: (6897553, 12)
             review_id  stars  useful  funny  cool  is_open  review_count  fans  friends_count  compliment_writer
KU_O5udG6zpxOg-VcAEodg      3       0      0     0        1          33.0   0.0             12                0.0
BiTunyQ73aT9WBnpR9DZGw      5       1      0     1        0          10.0   0.0            134                0.0
saUsX_uimxRlCVr67Z4Jig      3       0      0     0        1        1332.0  58.0            119               49.0
AqPFMleE6RsU23_auESxiA      5       1      0     1        1           9.0   0.0              0                0.0
Sx8TMOWLNuJBWer-0pcmoA      4       1      0     1        0         126.0   0.0              0                2.0


In [9]:
# Test Cell 2: Secondary feature extraction on filtered data only (memory-safe)
DATA_DIR = Path("./yelp_dataset")
TEST_NROWS = None #20_000

review_sample_for_filter = read_jsonl_columns(
    DATA_DIR / REVIEW_FILE,
    ["review_id", "user_id", "business_id", "date", "stars", "useful", "funny", "cool", "text"],
    nrows=TEST_NROWS,
)
review_filtered = filter_businesses_by_min_history_days(
    review_df=review_sample_for_filter,
    min_days=365,
)

secondary_features_sample = extract_secondary_features(
    data_dir=DATA_DIR,
    review_df=review_filtered,
    nrows=TEST_NROWS,
    early_window_days=30,
)

print("Reviews before filter:", len(review_sample_for_filter))
print("Reviews after filter:", len(review_filtered))
print("Secondary shape:", secondary_features_sample.shape)
secondary_preview_cols = [
    "review_id",
    "reviewer_bias",
    "early_rating_variance",
    "early_review_velocity",
    "user_tenure_days",
    "elite_persistence",
    "review_char_count",
    "review_word_count",
    "attribute_consistency",
]
print(secondary_features_sample[secondary_preview_cols].head(5).to_string(index=False))

Reviews before filter: 6990280
Reviews after filter: 6897553
Secondary shape: (6897553, 9)
             review_id  reviewer_bias  early_rating_variance  early_review_velocity  user_tenure_days  elite_persistence  review_char_count  review_word_count  attribute_consistency
KU_O5udG6zpxOg-VcAEodg          -1.06               1.286684               0.357143               906                  0                513                101                   True
BiTunyQ73aT9WBnpR9DZGw           0.70               0.408248               0.750000               301                  0                829                151                  False
saUsX_uimxRlCVr67Z4Jig          -1.69                    NaN               1.000000               518                  9                339                 55                   True
AqPFMleE6RsU23_auESxiA           0.22                    NaN               1.000000               351                  0                243                 40                  False

In [10]:
# Test Cell 3: Combine previous results and cache to disk
combined_features_sample = primary_features_sample.merge(
    secondary_features_sample,
    on="review_id",
    how="left",
    validate="1:1",
)

output_dir = Path("./test")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "combined_features_sample.csv"

combined_features_sample.to_csv(output_path, index=False)
print("Combined shape:", combined_features_sample.shape)
print("Saved to:", output_path.resolve())

Combined shape: (6897553, 20)
Saved to: /Users/eukey/Work/docker-share/ubuntu2204-llm-mnt/cai5133-project/test/combined_features_sample.csv
